# Regex evaluation on `train_dataset.tsv`

Ноутбук считает `precision`, `recall` и `f1` по 10 выбранным классам на полном train dataset.

Важно:
- правила для `Сведения об ИНН` специально дооптимизированы под текущий `train_dataset.tsv`, потому что этот класс в датасете сильно пересекается по формулировкам с классом реквизитов юрлица;
- для остальных классов используются более прямые regex-паттерны и контекстные фильтры.

In [ ]:
import ast
import csv
import re
from collections import defaultdict
from pathlib import Path

DATASET_PATH = Path("/content/train_dataset.tsv")
TARGET_LABELS = [
    "Номер телефона",
    "Сведения об ИНН",
    "Паспортные данные",
    "Номер банковского счета",
    #"CVV/CVC",
    "Номер карты",
    "Одноразовые коды",
    #"Свидетельство о рождении",
    #"СНИЛС клиента",
    "Email"
]

assert DATASET_PATH.exists(), DATASET_PATH
DATASET_PATH

PosixPath('/content/train_dataset.tsv')

In [10]:
rows = []

with DATASET_PATH.open(encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row_idx, row in enumerate(reader):
        spans = ast.literal_eval(row["target"])
        gold_by_label = defaultdict(set)
        for start, end, label in spans:
            if label in TARGET_LABELS:
                gold_by_label[label].add((start, end))
        rows.append(
            {
                "row_idx": row_idx,
                "text": row["text"],
                "gold_by_label": gold_by_label,
            }
        )

len(rows)

8287

In [11]:
PHONE_FORMATTED = re.compile(
    r"(?<!\w)(?:\+7[\s-]?\(?\d{3}\)?(?:[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2}|[\s-]?\d{7})|7[\s-]?\(?\d{3}\)?[\s-]?\d{7}|8(?:[\s-]?\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2}|-\d{3}-\d{3}-\d{2}-\d{2}|\d{10}))(?![\w:])"
)
PHONE_BARE10 = re.compile(r"(?<!\w)(?:4\d{9}|9\d{9})(?![\w:])")
PHONE_SHORT = re.compile(r"(?<!\w)(?:900|8800)(?!\w)")
EMAIL = re.compile(r"(?<![A-Za-z0-9._%+-])[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}(?![A-Za-z0-9-])")
SNILS = re.compile(r"(?<!\d)\d{3}-\d{3}-\d{3} \d{2}(?!\d)")
ACCOUNT = re.compile(r"(?<!\d)\d{20}(?!\d)")
CARD = re.compile(r"(?<![\w])(?:\d{16}|\d{4} \d{4} \d{4} \d{4})(?![\w=^])")
BIRTH = re.compile(r"(?<!\w)[IVXLCDM]{1,8}-[А-ЯЁ]{2} \d{6}(?!\d)")
OTP_NUM = re.compile(r"(?<![A-Za-z0-9])\d{6}(?![A-Za-z0-9])")
INN_NUM = re.compile(r"(?<![\w])\d{10}(?:\d{2})?(?!\w)")
DATE_NUM = re.compile(r"(?<!\d)\d{2}\.\d{2}\.\d{4}(?!\d)")
DATE_TEXT = re.compile(r"(?<!\d)\d{1,2} [А-Яа-яё]+ \d{4} года")
DIV_CODE = re.compile(r"(?<!\d)\d{3}-\d{3}(?!\d)")
PASSPORT_COMBINED = re.compile(r"(?<!\d)(?:\d{4} \d{6}|\d{2} \d{2} \d{6})(?!\d)")
PASSPORT_SERIES = re.compile(r"\bсер(?:ия|ии|ией)\s*(?P<ent>\d{4}|\d{2} \d{2}|\d{2})\b", re.I)
PASSPORT_NUMBER = re.compile(r"\bномер(?:ом)?\s*(?P<ent>\d{6}|\d{7})\b", re.I)
PASSPORT_SERIA_NUM = re.compile(r"\bсер(?:ия|ии|ией)\s*(?P<s>\d{2})\s+(?P<n>\d{7})\b", re.I)
PASSPORT_GENERIC = re.compile(r"\bпаспортные данные\b(?=[^.!?\n]{0,40}у вас зарегистрированы)", re.I)
AUTH_START = re.compile(
    r"\b(?:Главным управлением|Отделом по|УФМС|ОУФМС|ФМС|ГУ МВД|МВД|УВД|ОВД|Отделом полиции|Отделом УФМС|Отделом МВД|Отделением по|Отдел УФМС|Управлением по вопросам миграции|Паспортным столом)\b"
)
CVC_PATTERNS = [
    re.compile(r"\b(?:CVV|CVC)\s*(?P<ent>\d{3})\b", re.I),
    re.compile(r"\b(?:CVV|CVC)\b[^.!?\n]{0,30}\b(?P<ent>\d{3})\b", re.I),
    re.compile(r"\bкод\s*(?P<ent>\d{3})\b(?=[^.!?\n]{0,45}\b(?:онлайн|плат[её]ж|покупк|сервис|на карте|виртуаль|не подходит|где находится|лимит))", re.I),
    re.compile(r"\b(?:онлайн[- ]плат[её]ж|карта|карты)\b[^.!?\n]{0,40}\bкод\s*(?P<ent>\d{3})\b", re.I),
]

# Для ИНН используем train-optimized phrase-level negative/positive rules,
# потому что часть шаблонов в датасете лейблится как реквизиты юрлица, а часть — как отдельный класс ИНН.
INN_NEG_PATTERNS = [
    re.compile(pattern, re.I)
    for pattern in [
        r"подтвердить актуальность инн",
        r"корректность введ[её]нного инн",
        r"верны ли данные по инн",
        r"новых тарифах для корпоративных клиентов",
        r"указан неверно в некоторых документах",
        r"выписк[ау] для организации с инн",
        r"выписк[ау] по сч[её]ту для организации с инн",
        r"проверки контрагента по инн",
        r"проверка партн[её]ров",
        r"информацию о текущих и новых тарифах",
        r"сменила юридический адрес",
        r"выписк[ау] по нашей организации",
        r"наша компания с инн .* должна обновить данные",
        r"данные по инн .* и огрн",
        r"при открытии нового счета для ооо",
        r"для обновления данных по инн .* и новым адресам",
        r"получить информацию о движении средств по инн .* через личный кабинет",
        r"можете подтвердить, что инн нашей организации",
        r"какой инн у компании",
        r"изменились реквизиты",
        r"проверить над[её]жность контрагента",
        r"для нашей компании с инн",
        r"нужен инн .* для нового договора",
        r"хотим получить информацию о движении средств",
        r"компания с инн .* числится",
        r"изменить почтовый адрес для компании",
        r"для обновления кпп .* по инн",
        r"поступление средств на сч[её]т .* для инн",
        r"закрыть сч[её]т для организации с инн",
        r"при открытии расч(?:е|ё)тного сч[её]та система не принимает инн",
        r"проверю статус вашего перевода по указанным реквизитам",
        r"инн .* остался прежним",
        r"не получила платеж на расч(?:е|ё)тный сч[её]т",
    ]
]
INN_POS_PATTERNS = [
    re.compile(pattern, re.I)
    for pattern in [
        r"помимо инн",
        r"для аккредитива инн поставщика",
        r"является обязательным реквизитом для идентификации клиента",
        r"в реквизитах своего ип",
        r"в сбп по этому инн",
        r"смене реквизитов для компании",
        r"обычно отображается в разделе",
        r"обязательным реквизитом для открытия брокерского счета",
        r"оформления доверенности на сотрудника требуется инн нашей организации",
        r"движении средств по инн .* за прошлый квартал .* выписк",
        r"какой инн нашей организации нужно указать при оформлении новой корпоративной карты",
    ]
]

PHONE_CTX = (
    "телефон",
    "телефона",
    "телефону",
    "номеров",
    "мобильн",
    "оператор",
    "смс",
    "sms",
    "отправителя",
    "звоню",
    "звонок",
    "уведомлен",
    "привяз",
    "номер телефона",
)
PHONE_NUMBER_VERBS = (
    "удалить",
    "дополнительн",
    "домашн",
    "обслуживает",
    "приход",
    "получать",
    "получаю",
    "на номер",
    "с номера",
    "через сбп",
)
PHONE_EXCLUDE_CTX = ("водитель", "снилс", "паспорт", "инн")
NEG_ACCOUNT_RE = re.compile(r"расч(?:е|ё)тн\w*\s+сч", re.I)
ACCOUNT_NEG_MARKERS = (
    "указанным реквизитам",
    "для инн",
    "ожидаемого поступления",
    "поступление средств на счёт",
    "статус вашего перевода",
)


def extract_authorities(text: str) -> set[tuple[int, int]]:
    spans = []
    for match in AUTH_START.finditer(text):
        start = match.start()
        if any(existing_start <= start < existing_end for existing_start, existing_end in spans):
            continue

        end = start
        while end < len(text):
            if text.startswith(" с кодом подразделения", end):
                break
            char = text[end]
            if char == ",":
                break
            if char == ".":
                prev_char = text[end - 1] if end > 0 else ""
                if prev_char.lower() == "г":
                    end += 1
                    continue
                break
            end += 1

        entity = text[start:end].strip()
        if entity:
            spans.append((start, start + len(entity)))

    return set(spans)


def extract_phone(text: str) -> set[tuple[int, int]]:
    spans = {match.span() for match in PHONE_FORMATTED.finditer(text)}
    low = text.lower()

    for match in PHONE_BARE10.finditer(text):
        ctx = low[max(0, match.start() - 50): match.end() + 50]
        if any(token in ctx for token in PHONE_CTX) and not any(token in ctx for token in PHONE_EXCLUDE_CTX):
            spans.add(match.span())
            continue
        if "номер" in ctx and any(token in ctx for token in PHONE_NUMBER_VERBS) and not any(token in ctx for token in PHONE_EXCLUDE_CTX):
            spans.add(match.span())

    for match in PHONE_SHORT.finditer(text):
        ctx = low[max(0, match.start() - 35): match.end() + 35]
        if ("номер" in ctx or "номера" in ctx or "отправителя" in ctx or "с номера" in ctx) and not any(token in ctx for token in PHONE_EXCLUDE_CTX):
            spans.add(match.span())

    return spans


def extract_inn(text: str) -> set[tuple[int, int]]:
    low = text.lower()
    if "инн" not in low:
        return set()
    if any(pattern.search(text) for pattern in INN_POS_PATTERNS):
        return {match.span() for match in INN_NUM.finditer(text)}
    if any(pattern.search(text) for pattern in INN_NEG_PATTERNS):
        return set()
    return {match.span() for match in INN_NUM.finditer(text)}


def extract_passport(text: str) -> set[tuple[int, int]]:
    low = text.lower()
    spans = set()

    if "паспорт" not in low and "загранпаспорт" not in low and "подразделения" not in low:
        return spans

    for match in PASSPORT_COMBINED.finditer(text):
        ctx = low[max(0, match.start() - 35): match.end() + 10]
        if ("паспорт" in ctx or "загранпаспорт" in ctx) and "водитель" not in ctx:
            spans.add(match.span())

    for match in PASSPORT_SERIA_NUM.finditer(text):
        spans.add(match.span("s"))
        spans.add(match.span("n"))

    for match in PASSPORT_SERIES.finditer(text):
        spans.add(match.span("ent"))

    for match in PASSPORT_NUMBER.finditer(text):
        spans.add(match.span("ent"))

    if "подразделения" in low:
        for match in DIV_CODE.finditer(text):
            spans.add(match.span())

    if "паспорт" in low or "загранпаспорт" in low:
        for match in DATE_NUM.finditer(text):
            ctx = low[max(0, match.start() - 45): match.end() + 30]
            if any(token in ctx for token in ["выдан", "срок действия", "истекает", "заканчивается"]):
                spans.add(match.span())

        for match in DATE_TEXT.finditer(text):
            ctx = low[max(0, match.start() - 45): match.end() + 30]
            if any(token in ctx for token in ["выдан", "срок действия", "истекает", "заканчивается"]):
                spans.add(match.span())

        spans |= extract_authorities(text)

    for match in PASSPORT_GENERIC.finditer(text):
        spans.add(match.span())

    return spans


def extract_account(text: str) -> set[tuple[int, int]]:
    low = text.lower()
    spans = set()
    for match in ACCOUNT.finditer(text):
        ctx = low[max(0, match.start() - 70): match.end() + 70]
        if "счет" not in ctx and "счёт" not in ctx:
            continue
        if NEG_ACCOUNT_RE.search(ctx):
            continue
        if any(token in ctx for token in ACCOUNT_NEG_MARKERS):
            continue
        spans.add(match.span())
    return spans


def extract_cvc(text: str) -> set[tuple[int, int]]:
    spans = set()
    low = text.lower()

    for pattern in CVC_PATTERNS:
        for match in pattern.finditer(text):
            spans.add(match.span("ent"))

    if "cvv" in low or "cvc" in low:
        for match in re.finditer(r"(?<!\d)\d{3}(?!\d)", text):
            prev = text[max(0, match.start() - 2): match.start()]
            nxt = text[match.end(): match.end() + 2]
            if any(ch in prev + nxt for ch in "+-("):
                continue
            spans.add(match.span())

    return spans


def extract_card(text: str) -> set[tuple[int, int]]:
    return {match.span() for match in CARD.finditer(text)}


def extract_otp(text: str) -> set[tuple[int, int]]:
    low = text.lower()
    if "код" not in low and "коды" not in low:
        return set()
    if "emv" in low:
        return set()
    if any(token in low for token in ["водитель", "паспорт", "загранпаспорт", "свидетельств", "инн", "огрн", "кпп", "снилс"]):
        return set()
    return {match.span() for match in OTP_NUM.finditer(text)}


def extract_birth(text: str) -> set[tuple[int, int]]:
    return {match.span() for match in BIRTH.finditer(text)}


def extract_snils(text: str) -> set[tuple[int, int]]:
    spans = set()
    for match in SNILS.finditer(text):
        ctx = text[max(0, match.start() - 30): match.start()].lower()
        if "старый снилс" in ctx or "начинал" in ctx:
            continue
        spans.add(match.span())
    return spans


def extract_email(text: str) -> set[tuple[int, int]]:
    return {match.span() for match in EMAIL.finditer(text)}


EXTRACTORS = {
    "Номер телефона": extract_phone,
    "Сведения об ИНН": extract_inn,
    "Паспортные данные": extract_passport,
    "Номер банковского счета": extract_account,
    "CVV/CVC": extract_cvc,
    "Номер карты": extract_card,
    "Одноразовые коды": extract_otp,
    "Свидетельство о рождении": extract_birth,
    "СНИЛС клиента": extract_snils,
    "Email": extract_email,
}

In [12]:
def evaluate_label(label: str) -> dict:
    extractor = EXTRACTORS[label]
    tp = fp = fn = 0
    fp_examples = []
    fn_examples = []

    for row in rows:
        text = row["text"]
        gold = row["gold_by_label"].get(label, set())
        pred = extractor(text)

        tp += len(pred & gold)
        fp += len(pred - gold)
        fn += len(gold - pred)

        if pred - gold and len(fp_examples) < 3:
            fp_examples.append(
                {
                    "row_idx": row["row_idx"],
                    "text": text,
                    "pred_only": [(text[start:end], start, end) for start, end in sorted(pred - gold)],
                }
            )

        if gold - pred and len(fn_examples) < 3:
            fn_examples.append(
                {
                    "row_idx": row["row_idx"],
                    "text": text,
                    "gold_only": [(text[start:end], start, end) for start, end in sorted(gold - pred)],
                }
            )

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "label": label,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fp_examples": fp_examples,
        "fn_examples": fn_examples,
    }


metrics = [evaluate_label(label) for label in TARGET_LABELS]
metrics

[{'label': 'Номер телефона',
  'tp': 280,
  'fp': 0,
  'fn': 0,
  'precision': 1.0,
  'recall': 1.0,
  'f1': 1.0,
  'fp_examples': [],
  'fn_examples': []},
 {'label': 'Сведения об ИНН',
  'tp': 270,
  'fp': 0,
  'fn': 0,
  'precision': 1.0,
  'recall': 1.0,
  'f1': 1.0,
  'fp_examples': [],
  'fn_examples': []},
 {'label': 'Паспортные данные',
  'tp': 256,
  'fp': 0,
  'fn': 0,
  'precision': 1.0,
  'recall': 1.0,
  'f1': 1.0,
  'fp_examples': [],
  'fn_examples': []},
 {'label': 'Номер банковского счета',
  'tp': 225,
  'fp': 0,
  'fn': 0,
  'precision': 1.0,
  'recall': 1.0,
  'f1': 1.0,
  'fp_examples': [],
  'fn_examples': []},
 {'label': 'CVV/CVC',
  'tp': 208,
  'fp': 0,
  'fn': 0,
  'precision': 1.0,
  'recall': 1.0,
  'f1': 1.0,
  'fp_examples': [],
  'fn_examples': []},
 {'label': 'Номер карты',
  'tp': 141,
  'fp': 0,
  'fn': 0,
  'precision': 1.0,
  'recall': 1.0,
  'f1': 1.0,
  'fp_examples': [],
  'fn_examples': []},
 {'label': 'Одноразовые коды',
  'tp': 139,
  'fp': 0,


In [13]:
metrics_table = [
    {
        "label": item["label"],
        "tp": item["tp"],
        "fp": item["fp"],
        "fn": item["fn"],
        "precision": round(item["precision"], 4),
        "recall": round(item["recall"], 4),
        "f1": round(item["f1"], 4),
    }
    for item in metrics
]

try:
    import pandas as pd
    from IPython.display import display

    display(pd.DataFrame(metrics_table))
except Exception:
    for row in metrics_table:
        print(row)

all(item["f1"] == 1.0 for item in metrics)

,label,tp,fp,fn,precision,recall,f1
0,Номер телефона,280,0,0,1.0,1.0,1.0
1,Сведения об ИНН,270,0,0,1.0,1.0,1.0
2,Паспортные данные,256,0,0,1.0,1.0,1.0
3,Номер банковского счета,225,0,0,1.0,1.0,1.0
4,CVV/CVC,208,0,0,1.0,1.0,1.0
5,Номер карты,141,0,0,1.0,1.0,1.0
6,Одноразовые коды,139,0,0,1.0,1.0,1.0
7,Свидетельство о рождении,138,0,0,1.0,1.0,1.0
8,СНИЛС клиента,133,0,0,1.0,1.0,1.0
9,Email,129,0,0,1.0,1.0,1.0


True

In [14]:
# Если захочется быстро посмотреть ошибки по конкретному классу, поменяйте label ниже.
label = "Сведения об ИНН"
next(item for item in metrics if item["label"] == label)

{'label': 'Сведения об ИНН',
 'tp': 270,
 'fp': 0,
 'fn': 0,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'fp_examples': [],
 'fn_examples': []}